# Phase 2 — Data Cleaning & Preprocessing

This notebook implements conservative cleaning based on the executed Phase 1 notebook and its five reports. The objective is to make source types and labels consistent, preserve legitimate transactions, and expose unresolved quality limitations before exploratory analysis.

**Scope:** cleaning and validation only. No SQL Server, Power BI, advanced EDA, visualizations, machine learning, or Phase 3 work. Raw CSVs and raw DataFrames remain unchanged. Run all cells using the existing **E-Commerce Sales BI (.venv)** kernel. Reusable functions are in `src/data_processing/cleaning.py`; their decisions are recorded in `reports/cleaning_summary.csv`.

## A. Load Raw Data

Locate the project using pathlib, review the Phase 1 evidence, and load all nine CSVs. Start with strings so identifiers and ZIP prefixes retain their original representation. Only empty CSV fields are initially missing: literal review content such as `NA` or `NULL` must remain text. Phase 1 used Pandas' default null recognition; the current download has no extra literal null tokens, so this policy does not change its initial missing counts.

Capture hashes for the original files and DataFrame values. The pipeline makes independent deep copies before editing. The original DataFrames stay available in `raw_datasets` throughout the notebook.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in [current, *current.parents]
     if (folder / "src" / "config.py").is_file() and (folder / "data" / "raw").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the project folder or a subfolder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, REPORTS_DIR
from src.data_processing.cleaning import (
    DATASET_FILES, DATE_COLUMNS, NUMERIC_COLUMNS, add_validation_actions,
    clean_datasets, export_and_verify, file_hash, frame_hash, load_raw_data,
    markdown_table, missing_before_after, validate_integrity,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | NumPy {np.__version__}")
print(f"Environment: {Path(sys.prefix).name}")
display(Markdown((REPORTS_DIR / "phase_1_summary.md").read_text(encoding="utf-8")))
phase_1_reports = {
    name: pd.read_csv(REPORTS_DIR / f"{name}.csv") for name in [
        "dataset_overview", "missing_values_report", "duplicate_summary", "relationship_validation"
    ]
}
for name, report in phase_1_reports.items():
    print(f"Reviewed {name}: {len(report)} diagnostic rows")

Python 3.13.9 | pandas 3.1.0 | NumPy 2.5.4
Environment: .venv


# Phase 1 — Data Understanding Summary

Computed from all nine local raw CSVs by `notebooks/01_data_understanding.ipynb`. Counts and diagnostics use the complete tables; head previews show five rows only. No cleaning or source-data modification was performed.

## Dataset overview

| dataset | rows | columns | memory_mib |
| --- | --- | --- | --- |
| customers | 99441 | 5 | 30.95 |
| geolocation | 1000163 | 5 | 173.26 |
| orders | 99441 | 8 | 52.94 |
| order_items | 112650 | 7 | 40.5 |
| order_payments | 103886 | 5 | 16.23 |
| order_reviews | 99224 | 7 | 39.12 |
| products | 32951 | 9 | 6.31 |
| sellers | 3095 | 4 | 0.72 |
| category_translation | 71 | 2 | 0.01 |

## Missing values

| dataset | column | missing_count | missing_percentage |
| --- | --- | --- | --- |
| order_reviews | review_comment_title | 87656 | 88.34% |
| order_reviews | review_comment_message | 58247 | 58.70% |
| orders | order_delivered_customer_date | 2965 | 2.98% |
| products | product_name_lenght | 610 | 1.85% |
| products | product_category_name | 610 | 1.85% |
| products | product_description_lenght | 610 | 1.85% |
| products | product_photos_qty | 610 | 1.85% |
| orders | order_delivered_carrier_date | 1783 | 1.79% |
| orders | order_approved_at | 160 | 0.16% |
| products | product_weight_g | 2 | 0.01% |
| products | product_length_cm | 2 | 0.01% |
| products | product_height_cm | 2 | 0.01% |
| products | product_width_cm | 2 | 0.01% |

Whitespace-only text fields: 29. Percentages use each source table's row count. Missing values use Pandas' default CSV null recognition.

## Exact duplicates

| dataset | duplicate_rows_excess | duplicate_rows_all |
| --- | --- | --- |
| customers | 0 | 0 |
| geolocation | 261831 | 390005 |
| orders | 0 | 0 |
| order_items | 0 | 0 |
| order_payments | 0 | 0 |
| order_reviews | 0 | 0 |
| products | 0 | 0 |
| sellers | 0 | 0 |
| category_translation | 0 | 0 |

Excess counts exclude the first occurrence; all-duplicate counts include every row in duplicate groups. Repeated identifiers alone are not exact duplicate rows.

## Candidate keys

| dataset | key_columns | null_key_rows | duplicate_rows_excess | valid_primary_key |
| --- | --- | --- | --- | --- |
| customers | customer_id | 0.0 | 0 | True |
| orders | order_id | 0.0 | 0 | True |
| order_items | order_id + order_item_id | 0.0 | 0 | True |
| order_payments | order_id + payment_sequential | 0.0 | 0 | True |
| order_reviews | review_id | 0.0 | 814 | False |
| order_reviews | review_id + order_id | 0.0 | 0 | True |
| products | product_id | 0.0 | 0 | True |
| sellers | seller_id | 0.0 | 0 | True |
| category_translation | product_category_name | 0.0 | 0 | True |

Candidate keys are accepted only if unique and non-null. A failed hypothesis requires review, not automatic deletion. The complete duplicate CSV also includes intentionally repeatable identifiers.

## Customer identity

| customer_records | distinct_customer_id | distinct_customer_unique_id | identities_with_multiple_customer_records | maximum_records_per_identity |
| --- | --- | --- | --- | --- |
| 99441 | 99441 | 96096 | 2997 | 17 |

customer_id joins order-linked customer records to orders. customer_unique_id groups customer records across purchases; it must not be used as a unique key for the customer table.

## Relationships

| child_dataset | foreign_key | parent_dataset | null_foreign_key_rows | unmatched_child_rows | parent_keys_without_child | observed_parent_to_child |
| --- | --- | --- | --- | --- | --- | --- |
| orders | customer_id | customers | 0 | 0 | 0 | one-to-one |
| order_items | order_id | orders | 0 | 0 | 775 | one-to-many |
| order_items | product_id | products | 0 | 0 | 0 | one-to-many |
| order_items | seller_id | sellers | 0 | 0 | 0 | one-to-many |
| order_payments | order_id | orders | 0 | 0 | 1 | one-to-many |
| order_reviews | order_id | orders | 0 | 0 | 768 | one-to-many |
| products | product_category_name | category_translation | 610 | 13 | 0 | one-to-many |
| customers | customer_zip_code_prefix | geolocation | 0 | 278 | 4178 | many-to-many |
| sellers | seller_zip_code_prefix | geolocation | 0 | 7 | 16776 | many-to-many |

Cardinality is parent-to-child for matched values in this download. Parent keys without children are coverage gaps, not automatically invalid records. Null child keys and unmatched non-null keys are reported separately. ZIP lookups are repeated observations; direct joins can multiply rows. Joining items, payments and reviews at order grain can also multiply rows.

### Unmatched non-null foreign keys

- products.product_category_name → category_translation.product_category_name: 13 rows, 2 distinct values; examples: ["portateis_cozinha_e_preparadores_de_alimentos", "pc_gamer"].

- customers.customer_zip_code_prefix → geolocation.geolocation_zip_code_prefix: 278 rows, 157 distinct values; examples: ["70686", "72005", "71919", "73255", "72300", "73369", "71884", "72002", "73401", "71676"].

- sellers.seller_zip_code_prefix → geolocation.geolocation_zip_code_prefix: 7 rows, 7 distinct values; examples: ["82040", "91901", "72580", "02285", "07412", "71551", "37708"].

## Timestamp inspection

| dataset | column | missing_count | exact_format_mismatches | parse_failures_non_missing | minimum_parsed | maximum_parsed |
| --- | --- | --- | --- | --- | --- | --- |
| orders | order_purchase_timestamp | 0 | 0 | 0 | 2016-09-04 21:15:19 | 2018-10-17 17:30:18 |
| orders | order_approved_at | 160 | 0 | 0 | 2016-09-15 12:16:38 | 2018-09-03 17:40:06 |
| orders | order_delivered_carrier_date | 1783 | 0 | 0 | 2016-10-08 10:34:01 | 2018-09-11 19:48:28 |
| orders | order_delivered_customer_date | 2965 | 0 | 0 | 2016-10-11 13:46:32 | 2018-10-17 13:22:46 |
| orders | order_estimated_delivery_date | 0 | 0 | 0 | 2016-09-30 00:00:00 | 2018-11-12 00:00:00 |
| order_items | shipping_limit_date | 0 | 0 | 0 | 2016-09-19 00:15:34 | 2020-04-09 22:35:08 |
| order_reviews | review_creation_date | 0 | 0 | 0 | 2016-10-02 00:00:00 | 2018-08-31 00:00:00 |
| order_reviews | review_answer_timestamp | 0 | 0 | 0 | 2016-10-07 18:32:28 | 2018-10-29 12:27:35 |

Dates were parsed into separate diagnostic Series with format %Y-%m-%d %H:%M:%S. Original DataFrame columns remain strings. Timezone information is absent from the inspected format; no timezone conversion was applied.

### Missing milestones by order status

| order_status | orders | order_approved_at_missing | order_delivered_carrier_date_missing | order_delivered_customer_date_missing |
| --- | --- | --- | --- | --- |
| approved | 2 | 0 | 2 | 2 |
| canceled | 625 | 141 | 550 | 619 |
| created | 5 | 5 | 5 | 5 |
| delivered | 96478 | 14 | 2 | 8 |
| invoiced | 314 | 0 | 314 | 314 |
| processing | 301 | 0 | 301 | 301 |
| shipped | 1107 | 0 | 0 | 1107 |
| unavailable | 609 | 0 | 609 | 609 |

### Chronological flags

| dataset | expected_earlier | expected_later | comparable_rows | later_precedes_earlier_rows |
| --- | --- | --- | --- | --- |
| orders | order_purchase_timestamp | order_approved_at | 99281 | 0 |
| orders | order_purchase_timestamp | order_delivered_carrier_date | 97658 | 166 |
| orders | order_purchase_timestamp | order_delivered_customer_date | 96476 | 0 |
| orders | order_approved_at | order_delivered_carrier_date | 97644 | 1359 |
| orders | order_delivered_carrier_date | order_delivered_customer_date | 96475 | 23 |
| order_reviews | review_creation_date | review_answer_timestamp | 99224 | 0 |

These are investigation flags, not proven corrections. Compare process semantics before selecting a Phase 2 rule.

## Other initial numeric checks

| dataset | column | negative_count | zero_count |
| --- | --- | --- | --- |
| products | product_weight_g | 0 | 4 |
| products | product_length_cm | 0 | 0 |
| products | product_height_cm | 0 | 0 |
| products | product_width_cm | 0 | 0 |
| order_items | price | 0 | 0 |
| order_items | freight_value | 0 | 383 |
| order_payments | payment_value | 0 | 9 |
| order_payments | payment_installments | 0 | 2 |

Geolocation rows outside global latitude/longitude bounds: 0.

Zero freight or payment amounts may be valid. Non-positive product measurements require investigation. Source column spellings such as product_name_lenght and product_description_lenght are retained.

## Phase 2 investigation requirements

- Define a missing-value rule for order_reviews.review_comment_title: 87,656 missing (88.34%). Consider optional fields and order status before imputing.

- Define a missing-value rule for order_reviews.review_comment_message: 58,247 missing (58.70%). Consider optional fields and order status before imputing.

- Define a missing-value rule for orders.order_delivered_customer_date: 2,965 missing (2.98%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_name_lenght: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_category_name: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_description_lenght: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_photos_qty: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for orders.order_delivered_carrier_date: 1,783 missing (1.79%). Consider optional fields and order status before imputing.

- Define a missing-value rule for orders.order_approved_at: 160 missing (0.16%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_weight_g: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_length_cm: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_height_cm: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_width_cm: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Investigate geolocation: 261,831 excess exact duplicate rows; do not automatically deduplicate geographic observations.

- Review the order_reviews candidate key (review_id): 814 excess duplicate keys and 0 null-key rows.

- Plan unmatched-key handling for products.product_category_name: 13 rows do not match category_translation.

- Plan unmatched-key handling for customers.customer_zip_code_prefix: 278 rows do not match geolocation.

- Plan unmatched-key handling for sellers.seller_zip_code_prefix: 7 rows do not match geolocation.

- Investigate 166 orders rows where order_delivered_carrier_date precedes order_purchase_timestamp.

- Investigate 1,359 orders rows where order_delivered_carrier_date precedes order_approved_at.

- Investigate 23 orders rows where order_delivered_customer_date precedes order_delivered_carrier_date.

- Establish a documented geolocation lookup grain before joining ZIP prefixes; preserve leading zeros and investigate text variants before standardizing them.

- Define valid ranges for product measurements and payment-related fields before making corrections; retain source spelling until any rename mapping is documented.

- Preserve valid order-item and payment sequences and legitimate repeated customer identities. Resolve review-key assumptions before downstream modeling.

## Execution and scope

All cells reached report generation. Four CSV reports and this summary were generated and read back for validation. Raw-file SHA-256 hashes were checked before and after execution. No business metrics, visualizations, SQL, Power BI, machine learning, or Phase 2 transformations were implemented. Phase 1 is complete; wait for approval before Phase 2.


Reviewed dataset_overview: 9 diagnostic rows
Reviewed missing_values_report: 52 diagnostic rows
Reviewed duplicate_summary: 23 diagnostic rows
Reviewed relationship_validation: 9 diagnostic rows


In [2]:
raw_file_hashes = {filename: file_hash(RAW_DATA_DIR / filename) for filename in DATASET_FILES.values()}
manifest = json.loads((PROJECT_ROOT / "data" / "DATASET_MANIFEST.json").read_text(encoding="utf-8"))
for entry in manifest:
    assert raw_file_hashes[entry["filename"]] == entry["sha256"], entry["filename"]
raw_datasets = load_raw_data(RAW_DATA_DIR)
raw_frame_hashes = {name: frame_hash(frame) for name, frame in raw_datasets.items()}
display(pd.DataFrame([
    {"dataset": name, "rows": len(frame), "source_columns": frame.shape[1]}
    for name, frame in raw_datasets.items()
]))
assert len(raw_datasets) == 9

,dataset,rows,source_columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,category_translations,71,2
8,geolocation,1000163,5


## B. Data Type Cleaning

`clean_datasets()` runs the documented pipeline once, on independent copies. Sections C–F inspect the decisions and flags from that run. All timestamp fields use `%Y-%m-%d %H:%M:%S`. IDs, item/payment sequences, and ZIP prefixes remain strings. Numerical columns are explicitly parsed without rounding; integer-valued installment counts and review scores use nullable integers when valid.

Unparseable non-empty values become missing only in the processed typed column, with their original text retained in the row-level flag report and unchanged raw data. No invalid values were assumed in advance. Missing dates are never inferred, and no timezone is invented.

In [3]:
cleaned_datasets, cleaning_actions, data_quality_flags = clean_datasets(raw_datasets)
for name in raw_datasets:
    assert cleaned_datasets[name] is not raw_datasets[name]

type_actions = cleaning_actions.loc[cleaning_actions["action"].isin(["parse_datetime", "parse_numeric"])]
display(type_actions[["dataset", "column", "action", "affected_records", "details"]])
display(pd.DataFrame([
    {"dataset": name, "column": column, "cleaned_dtype": str(cleaned_datasets[name][column].dtype)}
    for name in raw_datasets
    for column in DATE_COLUMNS.get(name, []) + NUMERIC_COLUMNS.get(name, [])
]))

,dataset,column,action,affected_records,details
27,orders,order_purchase_timestamp,parse_datetime,99441,"{""invalid"": 0}"
29,orders,order_approved_at,parse_datetime,99281,"{""invalid"": 0}"
31,orders,order_delivered_carrier_date,parse_datetime,97658,"{""invalid"": 0}"
33,orders,order_delivered_customer_date,parse_datetime,96476,"{""invalid"": 0}"
35,orders,order_estimated_delivery_date,parse_datetime,99441,"{""invalid"": 0}"
51,order_items,shipping_limit_date,parse_datetime,112650,"{""invalid"": 0}"
53,order_items,price,parse_numeric,112650,"{""invalid"": 0}"
55,order_items,freight_value,parse_numeric,112650,"{""invalid"": 0}"
67,payments,payment_installments,parse_numeric,103886,"{""invalid"": 0}"
69,payments,payment_value,parse_numeric,103886,"{""invalid"": 0}"


,dataset,column,cleaned_dtype
0,orders,order_purchase_timestamp,datetime64[us]
1,orders,order_approved_at,datetime64[us]
2,orders,order_delivered_carrier_date,datetime64[us]
3,orders,order_delivered_customer_date,datetime64[us]
4,orders,order_estimated_delivery_date,datetime64[us]
5,order_items,shipping_limit_date,datetime64[us]
6,order_items,price,Float64
7,order_items,freight_value,Float64
8,payments,payment_installments,Int64
9,payments,payment_value,Float64


## C. Missing Values and Category Decisions

- **Orders:** preserve absent approvals and delivery milestones. Flag missing milestones separately for delivered orders; other statuses can explain absence. Never infer delivery dates.
- **Reviews:** keep records without titles/messages. Trim only outer whitespace; blank optional text becomes missing. Preserve case, accents, punctuation, line breaks and inner spacing.
- **Products:** fill missing category labels with the explicit sentinel `Unknown`. Add English category labels from the supplied translation table only; unmapped English labels also use `Unknown`, with separate flags for originally missing and untranslated categories. Do not add invented translations or a synthetic category row.
- **Attributes:** preserve missing dimensions, weights, photo quantities and text lengths. Preserve zero/non-positive values and flag them for investigation.

The missingness comparison covers the original source columns; added English labels, lookup diagnostics and Boolean quality flags are documented separately. Source column spellings are retained.

In [4]:
missing_report = missing_before_after(raw_datasets, cleaned_datasets)
display(missing_report.loc[(missing_report["missing_before"] > 0) | (missing_report["missing_after"] > 0)])
orders = cleaned_datasets["orders"]
milestone_columns = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
milestone_context = orders.groupby("order_status", dropna=False).size().rename("orders").to_frame()
for column in milestone_columns:
    milestone_context[f"missing_{column}"] = orders[column].isna().groupby(orders["order_status"], dropna=False).sum()
display(milestone_context)
display(cleaning_actions.loc[cleaning_actions["action"].isin(["missing_category_label", "english_category_lookup"])])

,dataset,column,rows_before,rows_after,dtype_before,dtype_after,missing_before,missing_after,missing_change,missing_percentage_before,missing_percentage_after,unknown_labels_after
9,orders,order_approved_at,99441,99441,string,datetime64[us],160,160,0,0.160899,0.160899,0
10,orders,order_delivered_carrier_date,99441,99441,string,datetime64[us],1783,1783,0,1.793023,1.793023,0
11,orders,order_delivered_customer_date,99441,99441,string,datetime64[us],2965,2965,0,2.981668,2.981668,0
28,reviews,review_comment_title,99224,99224,string,string,87656,87658,2,88.341530,88.343546,0
29,reviews,review_comment_message,99224,99224,string,string,58247,58274,27,58.702532,58.729743,0
33,products,product_category_name,32951,32951,string,string,610,0,-610,1.851234,0.000000,610
34,products,product_name_lenght,32951,32951,string,Float64,610,610,0,1.851234,1.851234,0
35,products,product_description_lenght,32951,32951,string,Float64,610,610,0,1.851234,1.851234,0
36,products,product_photos_qty,32951,32951,string,Float64,610,610,0,1.851234,1.851234,0
37,products,product_weight_g,32951,32951,string,Float64,2,2,0,0.006070,0.006070,0


,orders,missing_order_approved_at,missing_order_delivered_carrier_date,missing_order_delivered_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


,dataset,action,column,affected_records,decision,reason,details
153,products,missing_category_label,product_category_name,610,changed,"Unknown is a missingness label, not an inferre...",{}
154,products,english_category_lookup,product_category_name_english,32328,changed,Map only publisher-provided translations; miss...,"{""unknown_labels"": 623}"


## D. Duplicate Handling and ZIP Lookup

Only exact duplicates across all source geolocation columns are removed from `geolocation_clean.csv`. These are repeated identical geographic observations, not transactions. All raw observations remain in the raw file and original DataFrame. Other tables keep every row, including legitimate repeated orders and review IDs; review uniqueness is validated at `(review_id, order_id)`.

For each non-null ZIP prefix, the lookup uses distinct observations and chooses the most frequent **state/city pair**. Ties use state then city alphabetically. Within that pair, calculate median latitude and longitude and select the **existing coordinate observation** closest to that median using squared degree differences. Ties use latitude then longitude. This heuristic produces a representative observed location; it is not a precise address or travel-distance calculation. Invalid/missing coordinates are excluded from coordinate selection only; a prefix with no valid representative remains in the lookup with missing coordinates.

The lookup retains source-row counts, distinct-observation counts and city/state ambiguity flags. Alternative spellings and accents remain unchanged. Use this lookup for ZIP joins, not the repeated observations table.

In [5]:
duplicate_decisions = cleaning_actions.loc[cleaning_actions["action"] == "exact_duplicate_handling"]
display(duplicate_decisions[["dataset", "affected_records", "decision", "details"]])
geolocation_investigation = cleaning_actions.loc[cleaning_actions["action"].isin([
    "repeated_zip_prefix_observations", "repeated_coordinate_pairs_within_zip"
])]
display(geolocation_investigation[["action", "affected_records", "details"]])
zip_lookup = cleaned_datasets["geolocation_zip_lookup"]
display(zip_lookup.head())
print(f"ZIP lookup rows: {len(zip_lookup):,}; unique prefixes: {zip_lookup['zip_code_prefix'].nunique():,}")
display(zip_lookup[["distinct_observations", "distinct_coordinate_pairs", "coordinate_repetitions_excess", "distinct_city_labels", "distinct_states"]].describe())

,dataset,affected_records,decision,details
155,customers,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
156,orders,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
157,order_items,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
158,payments,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
159,reviews,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
160,products,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
161,sellers,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
162,category_translations,0,preserved,"{""raw_exact_duplicates"": 0, ""processed_exact_d..."
163,geolocation,261831,removed from processed copy,"{""raw_exact_duplicates"": 261831, ""processed_ex..."


,action,affected_records,details
164,repeated_zip_prefix_observations,737140,"{""prefixes_with_multiple_observations"": 17823,..."
165,repeated_coordinate_pairs_within_zip,18178,{}


,zip_code_prefix,geolocation_state,geolocation_city,geolocation_lat,geolocation_lng,distinct_observations,distinct_city_labels,distinct_states,distinct_coordinate_pairs,coordinate_repetitions_excess,source_rows,dq_conflicting_city_labels,dq_conflicting_states,dq_missing_coordinate
0,01001,SP,sao paulo,-23.549825,-46.63397,11,2,1,10,1,26,True,False,False
1,01002,SP,sao paulo,-23.548318,-46.635421,6,2,1,6,0,13,True,False,False
2,01003,SP,sao paulo,-23.548976,-46.635322,11,2,1,10,1,17,True,False,False
3,01004,SP,sao paulo,-23.549513,-46.634719,14,2,1,14,0,22,True,False,False
4,01005,SP,sao paulo,-23.549612,-46.636532,13,2,1,11,2,25,True,False,False


ZIP lookup rows: 19,015; unique prefixes: 19,015


,distinct_observations,distinct_coordinate_pairs,coordinate_repetitions_excess,distinct_city_labels,distinct_states
count,19015.000000,19015.000000,19015.000000,19015.000000,19015.000000
mean,38.828925,37.872942,0.955982,1.467578,1.000421
std,50.875735,49.998113,2.597769,0.538502,0.020508
min,1.000000,1.000000,0.000000,1.000000,1.000000
25%,8.000000,8.000000,0.000000,1.000000,1.000000
50%,23.000000,22.000000,0.000000,1.000000,1.000000
75%,49.000000,48.000000,1.000000,2.000000,1.000000
max,779.000000,746.000000,87.000000,5.000000,2.000000


## E. Text Cleaning

Strip leading/trailing whitespace from text fields, standardize category case for lookup matching, and replace whitespace-only text with missing values. No accent removal, translation inference, spelling correction, inner whitespace collapse, review lowercasing, or broad city-name normalization is applied. The action report counts affected records separately for each operation; these counts can overlap and should not be added as a count of unique changed rows.

In [6]:
text_actions = cleaning_actions.loc[cleaning_actions["action"].isin([
    "strip_outer_whitespace", "blank_text_to_missing", "normalize_category_case"
])]
display(text_actions.loc[text_actions["affected_records"] > 0, [
    "dataset", "column", "action", "affected_records", "reason"
]])

,dataset,column,action,affected_records,reason
76,reviews,review_comment_title,strip_outer_whitespace,1998,Trim edges only; preserve inner review content...
77,reviews,review_comment_title,blank_text_to_missing,2,Whitespace-only fields have no text content; s...
78,reviews,review_comment_message,strip_outer_whitespace,9451,Trim edges only; preserve inner review content...
79,reviews,review_comment_message,blank_text_to_missing,27,Whitespace-only fields have no text content; s...
140,geolocation,geolocation_city,strip_outer_whitespace,1,Trim edges only; preserve inner review content...


## F. Business Logic Validation and Quality Flags

Every checked rule appears in the action report, including rules with zero violations. Boolean `dq_*` columns identify affected records in processed files. The long flag report contains one row per record/rule, a source-record ordinal (one-based parsed CSV record, not physical text-line number), key values, observed values and the retention decision. Derived lookup rows have a ZIP key instead of a source-record ordinal. Multiple flags can describe the same record; informational flags are not necessarily errors.

Checks cover child-table coverage, missing milestones and delivered-order completeness, chronology, product measurements, financial values, review-key repetition, translation coverage, ZIP coverage/format and lookup ambiguity. Zero monetary amounts are retained. A shipping-limit year beyond the observed purchase-year range is a contextual flag, not a proven invalid deadline. Original financial values and failed-rule records are never silently corrected or deleted.

In [7]:
flag_summary = cleaning_actions.loc[cleaning_actions["decision"] == "flagged; retained", [
    "dataset", "action", "affected_records", "reason"
]].rename(columns={"action": "rule"})
display(flag_summary)
display(data_quality_flags.head(10))
print(f"Flag-report rows: {len(data_quality_flags):,}; overlapping record/rule observations are intentional.")

,dataset,rule,affected_records,reason
26,orders,invalid_order_purchase_timestamp,0,Unparseable source date; original text retaine...
28,orders,invalid_order_approved_at,0,Unparseable source date; original text retaine...
30,orders,invalid_order_delivered_carrier_date,0,Unparseable source date; original text retaine...
32,orders,invalid_order_delivered_customer_date,0,Unparseable source date; original text retaine...
34,orders,invalid_order_estimated_delivery_date,0,Unparseable source date; original text retaine...
50,order_items,invalid_shipping_limit_date,0,Unparseable source date; original text retaine...
52,order_items,invalid_price,0,Non-numeric source value; original text retain...
54,order_items,invalid_freight_value,0,Non-numeric source value; original text retain...
66,payments,invalid_payment_installments,0,Non-numeric source value; original text retain...
68,payments,invalid_payment_value,0,Non-numeric source value; original text retain...


,dataset,source_record_number,record_key,rule,severity,observed_values,decision,explanation
0,products,106,"{""product_id"": ""a41e356c76fab66334f36de622ecbd...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
1,products,129,"{""product_id"": ""d8dee61c2034d6d075997acef1870e...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
2,products,146,"{""product_id"": ""56139431d72cd51f19eb9f7dae4d16...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
3,products,155,"{""product_id"": ""46b48281eb6d663ced748f324108c7...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
4,products,198,"{""product_id"": ""5fb61f482620cb672f5e586bb132ea...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
5,products,245,"{""product_id"": ""e10758160da97891c2fdcbc35f0f03...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
6,products,295,"{""product_id"": ""39e3b9b12cd0bf8ee681bbc1c130fe...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
7,products,300,"{""product_id"": ""794de06c32a626a5692ff50e4985d3...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
8,products,348,"{""product_id"": ""7af3e2da474486a3519b0cba9dea8a...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...
9,products,429,"{""product_id"": ""629beb8e7317703dcc5f35b5463fd2...",category_missing,info,"{""product_category_name"": ""<NA>""}",retained; no inferred correction,Missing source category retained as an explici...


Flag-report rows: 20,111; overlapping record/rule observations are intentional.


## G. Data Integrity

Validate all primary/composite keys, including the unique ZIP lookup. Retain repeated `review_id` values and validate their review/order pairs. Compare source and processed row counts; only processed geolocation exact duplicates may be removed.

Verify core foreign keys and perform actual many-to-one joins with row-count checks. For product translation validation, treat `Unknown` from a missing source category as missingness, separate from non-null untranslated categories. Compare each price, freight and payment value using exact Decimal equality; do not rely only on totals. Confirm every raw DataFrame fingerprint remains unchanged.

In [8]:
row_counts, key_validation, relationship_validation, financial_validation = validate_integrity(raw_datasets, cleaned_datasets)
for name, frame in raw_datasets.items():
    assert frame_hash(frame) == raw_frame_hashes[name], f"Raw DataFrame changed: {name}"
display(row_counts)
display(key_validation)
display(relationship_validation)
display(financial_validation)
cleaning_summary = add_validation_actions(
    cleaning_actions, row_counts, key_validation, relationship_validation, financial_validation
)

,dataset,rows_before,rows_after,rows_removed
0,customers,99441,99441,0
1,orders,99441,99441,0
2,order_items,112650,112650,0
3,payments,103886,103886,0
4,reviews,99224,99224,0
5,products,32951,32951,0
6,sellers,3095,3095,0
7,category_translations,71,71,0
8,geolocation,1000163,738332,261831
9,geolocation_zip_lookup,0,19015,0


,dataset,key_columns,null_key_rows,duplicate_keys_excess,valid
0,customers,customer_id,0,0,True
1,orders,order_id,0,0,True
2,order_items,order_id + order_item_id,0,0,True
3,payments,order_id + payment_sequential,0,0,True
4,reviews,review_id + order_id,0,0,True
5,products,product_id,0,0,True
6,sellers,seller_id,0,0,True
7,category_translations,product_category_name,0,0,True
8,geolocation_zip_lookup,zip_code_prefix,0,0,True


,child_dataset,foreign_key,parent_dataset,parent_key,rows_after_join,child_rows,null_foreign_key_rows,matched_rows,unmatched_rows,unmatched_examples
0,orders,customer_id,customers,customer_id,99441,99441,0,99441,0,[]
1,order_items,order_id,orders,order_id,112650,112650,0,112650,0,[]
2,order_items,product_id,products,product_id,112650,112650,0,112650,0,[]
3,order_items,seller_id,sellers,seller_id,112650,112650,0,112650,0,[]
4,payments,order_id,orders,order_id,103886,103886,0,103886,0,[]
5,reviews,order_id,orders,order_id,99224,99224,0,99224,0,[]
6,customers,customer_zip_code_prefix,geolocation_zip_lookup,zip_code_prefix,99441,99441,0,99163,278,"[""72300"", ""11547"", ""64605"", ""72465"", ""07729"", ..."
7,sellers,seller_zip_code_prefix,geolocation_zip_lookup,zip_code_prefix,3095,3095,0,3088,7,"[""82040"", ""91901"", ""72580"", ""02285"", ""07412"", ..."
8,products,product_category_name,category_translations,product_category_name,32951,32951,610,32328,13,"[""pc_gamer"", ""portateis_cozinha_e_preparadores..."


,dataset,column,rows_checked,exact_decimal_equality
0,order_items,price,112650,True
1,order_items,freight_value,112650,True
2,payments,payment_value,103886,True


## H. Export and Verify Processed Data

Write the nine cleaned datasets plus the separate ZIP lookup to `data/processed/`, using UTF-8, blank fields for missing values and the same timestamp format throughout. CSV does not store types: the verification reloads using the in-memory schema, compares all values, rechecks monetary values exactly and confirms row counts and identifier representation. The returned export inventory records file hashes. Re-running overwrites these derived outputs only.

In [9]:
export_inventory = export_and_verify(cleaned_datasets, PROCESSED_DATA_DIR)
display(export_inventory)
assert len(export_inventory) == 10
assert export_inventory["round_trip_validated"].all()

report_frames = {
    "cleaning_summary.csv": cleaning_summary,
    "missing_values_before_after.csv": missing_report,
    "data_quality_flags.csv": data_quality_flags,
}
for filename, report in report_frames.items():
    path = REPORTS_DIR / filename
    report.to_csv(path, index=False, encoding="utf-8", lineterminator="\n")
    reloaded = pd.read_csv(path, keep_default_na=False)
    assert reloaded.shape == report.shape, f"Report export mismatch: {filename}"
    assert reloaded.columns.tolist() == report.columns.tolist()
    print(f"Verified report: {path.relative_to(PROJECT_ROOT)} ({len(report):,} rows)")

,dataset,filename,rows,columns,bytes,sha256,round_trip_validated
0,customers,customers_clean.csv,99441,7,9779318,dadd4167d59c0028dbce61ebd7282b6eb665ef776b79d7...,True
1,orders,orders_clean.csv,99441,27,28735749,495c9c11e5e638d4c576d1a406e51a2180b5565ab9b314...,True
2,order_items,order_items_clean.csv,112650,16,20994520,34037964980e1fdb300d64448eef8dc3e42903bfe67750...,True
3,payments,payments_clean.csv,103886,11,9373415,9441054e0298f27da1cb846cb8ac71378c32a1a855a8bd...,True
4,reviews,reviews_clean.csv,99224,14,18106274,1e9b833b7d2bc2420da63e63cdd836a741f7eb37a3fe82...,True
5,products,products_clean.csv,32951,24,6022937,2fe1d2904429ff2587266b690a26112446cd3ba4afe74b...,True
6,sellers,sellers_clean.csv,3095,6,201789,f12eef90d97555e2c3c0a4634a6ffe0aa277e1511a4a43...,True
7,category_translations,category_translations_clean.csv,71,2,2540,5233473346639d5ff3da9b784385ba284778abfaac1062...,True
8,geolocation,geolocation_clean.csv,738332,9,61477686,bb8cd9f785f156b091781677144c33ef7bbbc03fa06a7a...,True
9,geolocation_zip_lookup,geolocation_zip_lookup.csv,19015,14,1736166,f6bd7c09504f150764f8b0c10ad1288c0a92eee99735e1...,True


Verified report: reports\cleaning_summary.csv (252 rows)
Verified report: reports\missing_values_before_after.csv (52 rows)
Verified report: reports\data_quality_flags.csv (20,111 rows)


## I. Focused Automated Tests and Raw-File Integrity

Run regression tests for raw-data independence, missing/invalid values, review composite keys, review-text preservation, leading zeros, ZIP representative selection and ties, join multiplication, CSV round trips, and exact monetary preservation. These tests use small synthetic inputs and do not change project data. Finish by checking all raw-file hashes against the pre-run values and initialization manifest.

In [10]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_cleaning.py", "-q"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8", check=True,
)
print(test_result.stdout)
for filename, before_hash in raw_file_hashes.items():
    assert file_hash(RAW_DATA_DIR / filename) == before_hash, f"Raw file changed: {filename}"
    root_copy = PROJECT_ROOT / filename
    if root_copy.is_file():
        assert file_hash(root_copy) == before_hash
print("Raw-file hashes and in-memory source fingerprints are unchanged.")

...........                                                              [100%]
11 passed in 1.30s



Raw-file hashes and in-memory source fingerprints are unchanged.


## J. Phase 2 Summary and Limitations

Generate the final report from the measured cleaning actions and checks. Persistent flags describe source limitations; they are not a claim that all anomalies have been resolved. Stop after report generation and wait for approval before Phase 3.

In [11]:
changed_actions = cleaning_actions.loc[
    cleaning_actions["decision"].eq("changed") & cleaning_actions["affected_records"].gt(0),
    ["dataset", "action", "column", "affected_records"]
]
missing_display = missing_report.loc[
    missing_report["missing_before"].gt(0) | missing_report["missing_after"].gt(0),
    ["dataset", "column", "missing_before", "missing_after", "missing_change", "unknown_labels_after"]
]
active_flags = flag_summary.loc[flag_summary["affected_records"].gt(0), ["dataset", "rule", "affected_records"]]
invalid_conversion_flags = flag_summary.loc[flag_summary["rule"].str.startswith("invalid_") & ~flag_summary["rule"].eq("invalid_zip_format")]
conversion_failure_count = int(invalid_conversion_flags["affected_records"].sum())

summary_parts = [
    "# Phase 2 — Data Cleaning & Preprocessing Summary",
    "Generated by the executed cleaning notebook from all nine local raw datasets. Only processed outputs and Phase 2 reports were written. Raw files and source DataFrames are unchanged.",
    "## Cleaning methodology",
    "- Read source fields as strings, recognizing only empty fields as missing. Preserve identifiers, ZIP prefixes and item/payment sequence keys as strings.",
    "- Trim outer whitespace only; preserve review case, accents, punctuation, internal whitespace and line breaks. Convert blank optional text to missing. Normalize category case without inventing translations.",
    "- Parse timestamps with %Y-%m-%d %H:%M:%S and validate numeric types. No date, dimension, weight or financial amount is imputed. Invalid non-empty source values, if encountered, are retained in raw data and the row-level flag report.",
    "- Use Unknown for missing product category display labels and absent English labels. Separate missing-source-category and untranslated-category flags; keep the 71 publisher translation rows unchanged. Unknown is not a new translation or business category.",
    "- Keep all orders, items, payments, reviews, products, customers, sellers and translation rows. Remove only exact repeated geolocation observations from the processed copy. Do not deduplicate by order_id or review_id.",
    "## Before-and-after rows", markdown_table(row_counts),
    "## Measured transformations", markdown_table(changed_actions),
    "Action counts can overlap. Parsing affects non-missing source fields while preserving their values; it is a type change rather than an inferred business correction.",
    "## Duplicate decisions", markdown_table(duplicate_decisions[["dataset", "affected_records", "decision"]]),
    "## Missing-value comparison", markdown_table(missing_display),
    "Missing counts cover source columns. The new English display field also uses Unknown for untranslated or absent categories. Added Boolean quality flags preserve missing-source information when a display sentinel is used.",
    "## Missing milestones by order status", markdown_table(milestone_context.reset_index()),
    "## ZIP-prefix representative strategy",
    "Remove exact duplicate observations before representative selection. Choose the modal state/city pair per prefix; break count ties alphabetically by state then city. Within that pair, calculate median latitude/longitude and select an existing globally valid coordinate with minimum squared degree-distance to the median; break ties by latitude then longitude. Retain a prefix with missing coordinates if no valid representative exists. Alternative labels are preserved; city/state ambiguity remains flagged. This is an approximate representative, not a verified address or routing distance.",
    f"Lookup rows: {len(zip_lookup):,}. Every prefix is unique and non-null. Customer and seller many-to-one lookup joins preserved row counts.",
    markdown_table(geolocation_investigation[["action", "affected_records", "details"]]),
    "Coordinate repetitions are counted after exact-row deduplication: they represent the same ZIP and coordinates with different source labels. These distinct labeled observations are preserved. The lookup stores both distinct observation and coordinate-pair counts.",
    "## Business-rule and source-quality flags", markdown_table(active_flags),
    f"Conversion/coordinate validation flags (excluding ZIP format): {conversion_failure_count:,}. The complete action CSV also records all zero-count checks; the flag CSV contains only affected record/rule pairs. There are {len(data_quality_flags):,} flag-report rows, not that many distinct bad records.",
    "## Key validation", markdown_table(key_validation),
    "review_id alone is intentionally not treated as a primary key; the composite review/order key is validated. Geolocation observations have no invented natural primary key; the separate ZIP lookup does.",
    "## Relationships and join sizes", markdown_table(relationship_validation),
    "Missing source categories are separate from unmatched non-null categories. Unmatched ZIP/category keys are retained and flagged rather than fabricated. Relationship coverage limitations are not silently repaired. All tested many-to-one joins preserve their child-row counts.",
    "## Financial preservation", markdown_table(financial_validation),
    "Every original price, freight value and payment value is equal per row as an exact Decimal, both before export and on CSV read-back. Zero/negative rules flag values without modifying them.",
    "## Automated tests", test_result.stdout.strip(),
    "Tests cover raw-copy independence, invalid-date provenance, optional text, literal null words, leading-zero preservation, exact financial values, review composite keys, deterministic ZIP selection, conflicting labels, missing coordinates, row-preserving joins and export round trips.",
    "## Processed files", markdown_table(export_inventory),
    "CSV files have no embedded type metadata. Use the pipeline constants and verified in-memory dtypes when reloading; always read identifiers and ZIP prefixes as strings and parse date fields explicitly. Quality flags use True/False; missing values export as empty fields.",
    "## Report files",
    "- reports/cleaning_summary.csv: transformations, retained values, all rule counts, row/key/FK/join/financial validation details (structured JSON in details).",
    "- reports/missing_values_before_after.csv: missingness and dtypes for every original source column with explicit before/after denominators.",
    "- reports/data_quality_flags.csv: one row per affected record and rule, observed values, keys and retention decisions. Source record numbers refer to parsed CSV records (one-based), not physical lines; derived lookup rows use ZIP keys instead.",
    "- reports/phase_2_summary.md: this generated summary.",
    "## Remaining limitations and recommendations for Phase 3",
    "- Decide and document eligibility rules for delivery analyses using status, missing milestones and chronology flags; do not silently exclude records.",
    "- Treat optional missing review text separately from available ratings. Repeated review IDs can belong to different orders; respect the composite grain.",
    "- Use Unknown and translation flags explicitly when grouping products. No unsupported English translations, dimensions or weights were invented.",
    "- Use only the unique ZIP lookup for geographic joins. Retain unmatched entities and report the geographic coverage limitation. City variants and cross-state ZIP ambiguity remain unresolved; do not treat representative coordinates as precise customer locations.",
    "- Validate numeric ranges and zero-value business meanings before selecting analytic exclusions. Dates outside the observed purchase-year range remain contextual flags.",
    "- Keep item, payment and review grains separate before combining summaries, to prevent one-to-many join multiplication. Preserve all source transaction values.",
    "## Completion status",
    "The notebook executed through all cleaning, validation, exports, read-back checks, focused tests and source-integrity checks. Ten processed datasets and four reports were generated. All nine raw hashes match the initialization manifest; source DataFrames retain their original fingerprints. No SQL Server, Power BI, advanced EDA, visualizations, machine learning or Phase 3 work was implemented. Stop after Phase 2 and wait for approval.",
]
phase_2_summary = "\n\n".join(summary_parts) + "\n"
summary_path = REPORTS_DIR / "phase_2_summary.md"
summary_path.write_text(phase_2_summary, encoding="utf-8")
assert summary_path.read_text(encoding="utf-8") == phase_2_summary
display(Markdown(phase_2_summary))
print("Phase 2 complete. Stop here; Phase 3 requires approval.")

# Phase 2 — Data Cleaning & Preprocessing Summary

Generated by the executed cleaning notebook from all nine local raw datasets. Only processed outputs and Phase 2 reports were written. Raw files and source DataFrames are unchanged.

## Cleaning methodology

- Read source fields as strings, recognizing only empty fields as missing. Preserve identifiers, ZIP prefixes and item/payment sequence keys as strings.

- Trim outer whitespace only; preserve review case, accents, punctuation, internal whitespace and line breaks. Convert blank optional text to missing. Normalize category case without inventing translations.

- Parse timestamps with %Y-%m-%d %H:%M:%S and validate numeric types. No date, dimension, weight or financial amount is imputed. Invalid non-empty source values, if encountered, are retained in raw data and the row-level flag report.

- Use Unknown for missing product category display labels and absent English labels. Separate missing-source-category and untranslated-category flags; keep the 71 publisher translation rows unchanged. Unknown is not a new translation or business category.

- Keep all orders, items, payments, reviews, products, customers, sellers and translation rows. Remove only exact repeated geolocation observations from the processed copy. Do not deduplicate by order_id or review_id.

## Before-and-after rows

| dataset | rows_before | rows_after | rows_removed |
| --- | --- | --- | --- |
| customers | 99441 | 99441 | 0 |
| orders | 99441 | 99441 | 0 |
| order_items | 112650 | 112650 | 0 |
| payments | 103886 | 103886 | 0 |
| reviews | 99224 | 99224 | 0 |
| products | 32951 | 32951 | 0 |
| sellers | 3095 | 3095 | 0 |
| category_translations | 71 | 71 | 0 |
| geolocation | 1000163 | 738332 | 261831 |
| geolocation_zip_lookup | 0 | 19015 | 0 |

## Measured transformations

| dataset | action | column | affected_records |
| --- | --- | --- | --- |
| orders | parse_datetime | order_purchase_timestamp | 99441 |
| orders | parse_datetime | order_approved_at | 99281 |
| orders | parse_datetime | order_delivered_carrier_date | 97658 |
| orders | parse_datetime | order_delivered_customer_date | 96476 |
| orders | parse_datetime | order_estimated_delivery_date | 99441 |
| order_items | parse_datetime | shipping_limit_date | 112650 |
| order_items | parse_numeric | price | 112650 |
| order_items | parse_numeric | freight_value | 112650 |
| payments | parse_numeric | payment_installments | 103886 |
| payments | parse_numeric | payment_value | 103886 |
| reviews | strip_outer_whitespace | review_comment_title | 1998 |
| reviews | blank_text_to_missing | review_comment_title | 2 |
| reviews | strip_outer_whitespace | review_comment_message | 9451 |
| reviews | blank_text_to_missing | review_comment_message | 27 |
| reviews | parse_datetime | review_creation_date | 99224 |
| reviews | parse_datetime | review_answer_timestamp | 99224 |
| reviews | parse_numeric | review_score | 99224 |
| products | parse_numeric | product_name_lenght | 32341 |
| products | parse_numeric | product_description_lenght | 32341 |
| products | parse_numeric | product_photos_qty | 32341 |
| products | parse_numeric | product_weight_g | 32949 |
| products | parse_numeric | product_length_cm | 32949 |
| products | parse_numeric | product_height_cm | 32949 |
| products | parse_numeric | product_width_cm | 32949 |
| geolocation | strip_outer_whitespace | geolocation_city | 1 |
| geolocation | parse_numeric | geolocation_lat | 1000163 |
| geolocation | parse_numeric | geolocation_lng | 1000163 |
| products | missing_category_label | product_category_name | 610 |
| products | english_category_lookup | product_category_name_english | 32328 |
| geolocation_zip_lookup | representative_zip_lookup | zip_code_prefix | 19015 |

Action counts can overlap. Parsing affects non-missing source fields while preserving their values; it is a type change rather than an inferred business correction.

## Duplicate decisions

| dataset | affected_records | decision |
| --- | --- | --- |
| customers | 0 | preserved |
| orders | 0 | preserved |
| order_items | 0 | preserved |
| payments | 0 | preserved |
| reviews | 0 | preserved |
| products | 0 | preserved |
| sellers | 0 | preserved |
| category_translations | 0 | preserved |
| geolocation | 261831 | removed from processed copy |

## Missing-value comparison

| dataset | column | missing_before | missing_after | missing_change | unknown_labels_after |
| --- | --- | --- | --- | --- | --- |
| orders | order_approved_at | 160 | 160 | 0 | 0 |
| orders | order_delivered_carrier_date | 1783 | 1783 | 0 | 0 |
| orders | order_delivered_customer_date | 2965 | 2965 | 0 | 0 |
| reviews | review_comment_title | 87656 | 87658 | 2 | 0 |
| reviews | review_comment_message | 58247 | 58274 | 27 | 0 |
| products | product_category_name | 610 | 0 | -610 | 610 |
| products | product_name_lenght | 610 | 610 | 0 | 0 |
| products | product_description_lenght | 610 | 610 | 0 | 0 |
| products | product_photos_qty | 610 | 610 | 0 | 0 |
| products | product_weight_g | 2 | 2 | 0 | 0 |
| products | product_length_cm | 2 | 2 | 0 | 0 |
| products | product_height_cm | 2 | 2 | 0 | 0 |
| products | product_width_cm | 2 | 2 | 0 | 0 |

Missing counts cover source columns. The new English display field also uses Unknown for untranslated or absent categories. Added Boolean quality flags preserve missing-source information when a display sentinel is used.

## Missing milestones by order status

| order_status | orders | missing_order_approved_at | missing_order_delivered_carrier_date | missing_order_delivered_customer_date |
| --- | --- | --- | --- | --- |
| approved | 2 | 0 | 2 | 2 |
| canceled | 625 | 141 | 550 | 619 |
| created | 5 | 5 | 5 | 5 |
| delivered | 96478 | 14 | 2 | 8 |
| invoiced | 314 | 0 | 314 | 314 |
| processing | 301 | 0 | 301 | 301 |
| shipped | 1107 | 0 | 0 | 1107 |
| unavailable | 609 | 0 | 609 | 609 |

## ZIP-prefix representative strategy

Remove exact duplicate observations before representative selection. Choose the modal state/city pair per prefix; break count ties alphabetically by state then city. Within that pair, calculate median latitude/longitude and select an existing globally valid coordinate with minimum squared degree-distance to the median; break ties by latitude then longitude. Retain a prefix with missing coordinates if no valid representative exists. Alternative labels are preserved; city/state ambiguity remains flagged. This is an approximate representative, not a verified address or routing distance.

Lookup rows: 19,015. Every prefix is unique and non-null. Customer and seller many-to-one lookup joins preserved row counts.

| action | affected_records | details |
| --- | --- | --- |
| repeated_zip_prefix_observations | 737140 | {"prefixes_with_multiple_observations": 17823, "distinct_prefixes": 19015} |
| repeated_coordinate_pairs_within_zip | 18178 | {} |

Coordinate repetitions are counted after exact-row deduplication: they represent the same ZIP and coordinates with different source labels. These distinct labeled observations are preserved. The lookup stores both distinct observation and coordinate-pair counts.

## Business-rule and source-quality flags

| dataset | rule | affected_records |
| --- | --- | --- |
| products | category_missing | 610 |
| products | category_untranslated | 13 |
| orders | no_items | 775 |
| orders | no_payments | 1 |
| orders | no_reviews | 768 |
| orders | missing_order_approved_at | 160 |
| orders | delivered_missing_order_approved_at | 14 |
| orders | missing_order_delivered_carrier_date | 1783 |
| orders | delivered_missing_order_delivered_carrier_date | 2 |
| orders | missing_order_delivered_customer_date | 2965 |
| orders | delivered_missing_order_delivered_customer_date | 8 |
| orders | carrier_before_purchase | 166 |
| orders | carrier_before_approval | 1359 |
| orders | delivery_before_carrier | 23 |
| order_items | shipping_limit_after_purchase_year_range | 4 |
| products | missing_attributes | 611 |
| products | nonpositive_product_weight_g | 4 |
| order_items | zero_freight_value | 383 |
| payments | zero_payment_value | 9 |
| payments | nonpositive_installments | 2 |
| reviews | repeated_review_id | 1603 |
| customers | unmatched_zip | 278 |
| sellers | unmatched_zip | 7 |
| geolocation_zip_lookup | conflicting_city_labels | 8555 |
| geolocation_zip_lookup | conflicting_states | 8 |

Conversion/coordinate validation flags (excluding ZIP format): 0. The complete action CSV also records all zero-count checks; the flag CSV contains only affected record/rule pairs. There are 20,111 flag-report rows, not that many distinct bad records.

## Key validation

| dataset | key_columns | null_key_rows | duplicate_keys_excess | valid |
| --- | --- | --- | --- | --- |
| customers | customer_id | 0 | 0 | True |
| orders | order_id | 0 | 0 | True |
| order_items | order_id + order_item_id | 0 | 0 | True |
| payments | order_id + payment_sequential | 0 | 0 | True |
| reviews | review_id + order_id | 0 | 0 | True |
| products | product_id | 0 | 0 | True |
| sellers | seller_id | 0 | 0 | True |
| category_translations | product_category_name | 0 | 0 | True |
| geolocation_zip_lookup | zip_code_prefix | 0 | 0 | True |

review_id alone is intentionally not treated as a primary key; the composite review/order key is validated. Geolocation observations have no invented natural primary key; the separate ZIP lookup does.

## Relationships and join sizes

| child_dataset | foreign_key | parent_dataset | parent_key | rows_after_join | child_rows | null_foreign_key_rows | matched_rows | unmatched_rows | unmatched_examples |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| orders | customer_id | customers | customer_id | 99441 | 99441 | 0 | 99441 | 0 | [] |
| order_items | order_id | orders | order_id | 112650 | 112650 | 0 | 112650 | 0 | [] |
| order_items | product_id | products | product_id | 112650 | 112650 | 0 | 112650 | 0 | [] |
| order_items | seller_id | sellers | seller_id | 112650 | 112650 | 0 | 112650 | 0 | [] |
| payments | order_id | orders | order_id | 103886 | 103886 | 0 | 103886 | 0 | [] |
| reviews | order_id | orders | order_id | 99224 | 99224 | 0 | 99224 | 0 | [] |
| customers | customer_zip_code_prefix | geolocation_zip_lookup | zip_code_prefix | 99441 | 99441 | 0 | 99163 | 278 | ["72300", "11547", "64605", "72465", "07729", "72904", "35408", "78554", "73369", "08980"] |
| sellers | seller_zip_code_prefix | geolocation_zip_lookup | zip_code_prefix | 3095 | 3095 | 0 | 3088 | 7 | ["82040", "91901", "72580", "02285", "07412", "71551", "37708"] |
| products | product_category_name | category_translations | product_category_name | 32951 | 32951 | 610 | 32328 | 13 | ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"] |

Missing source categories are separate from unmatched non-null categories. Unmatched ZIP/category keys are retained and flagged rather than fabricated. Relationship coverage limitations are not silently repaired. All tested many-to-one joins preserve their child-row counts.

## Financial preservation

| dataset | column | rows_checked | exact_decimal_equality |
| --- | --- | --- | --- |
| order_items | price | 112650 | True |
| order_items | freight_value | 112650 | True |
| payments | payment_value | 103886 | True |

Every original price, freight value and payment value is equal per row as an exact Decimal, both before export and on CSV read-back. Zero/negative rules flag values without modifying them.

## Automated tests

...........                                                              [100%]
11 passed in 1.30s

Tests cover raw-copy independence, invalid-date provenance, optional text, literal null words, leading-zero preservation, exact financial values, review composite keys, deterministic ZIP selection, conflicting labels, missing coordinates, row-preserving joins and export round trips.

## Processed files

| dataset | filename | rows | columns | bytes | sha256 | round_trip_validated |
| --- | --- | --- | --- | --- | --- | --- |
| customers | customers_clean.csv | 99441 | 7 | 9779318 | dadd4167d59c0028dbce61ebd7282b6eb665ef776b79d7d0a000fbacb428575d | True |
| orders | orders_clean.csv | 99441 | 27 | 28735749 | 495c9c11e5e638d4c576d1a406e51a2180b5565ab9b31495ebcf72bbbedca6eb | True |
| order_items | order_items_clean.csv | 112650 | 16 | 20994520 | 34037964980e1fdb300d64448eef8dc3e42903bfe67750ecc2ecbd3a622b8770 | True |
| payments | payments_clean.csv | 103886 | 11 | 9373415 | 9441054e0298f27da1cb846cb8ac71378c32a1a855a8bdbb6c55fa971428e1c5 | True |
| reviews | reviews_clean.csv | 99224 | 14 | 18106274 | 1e9b833b7d2bc2420da63e63cdd836a741f7eb37a3fe82b3a5e5ec2867e2ea08 | True |
| products | products_clean.csv | 32951 | 24 | 6022937 | 2fe1d2904429ff2587266b690a26112446cd3ba4afe74bfe555029b56e823258 | True |
| sellers | sellers_clean.csv | 3095 | 6 | 201789 | f12eef90d97555e2c3c0a4634a6ffe0aa277e1511a4a43e4dc32a9fe6ffbca7e | True |
| category_translations | category_translations_clean.csv | 71 | 2 | 2540 | 5233473346639d5ff3da9b784385ba284778abfaac1062572abe99e8aba8b7e3 | True |
| geolocation | geolocation_clean.csv | 738332 | 9 | 61477686 | bb8cd9f785f156b091781677144c33ef7bbbc03fa06a7a14db0c51d8b9b2aec8 | True |
| geolocation_zip_lookup | geolocation_zip_lookup.csv | 19015 | 14 | 1736166 | f6bd7c09504f150764f8b0c10ad1288c0a92eee99735e1f94bf192536879172e | True |

CSV files have no embedded type metadata. Use the pipeline constants and verified in-memory dtypes when reloading; always read identifiers and ZIP prefixes as strings and parse date fields explicitly. Quality flags use True/False; missing values export as empty fields.

## Report files

- reports/cleaning_summary.csv: transformations, retained values, all rule counts, row/key/FK/join/financial validation details (structured JSON in details).

- reports/missing_values_before_after.csv: missingness and dtypes for every original source column with explicit before/after denominators.

- reports/data_quality_flags.csv: one row per affected record and rule, observed values, keys and retention decisions. Source record numbers refer to parsed CSV records (one-based), not physical lines; derived lookup rows use ZIP keys instead.

- reports/phase_2_summary.md: this generated summary.

## Remaining limitations and recommendations for Phase 3

- Decide and document eligibility rules for delivery analyses using status, missing milestones and chronology flags; do not silently exclude records.

- Treat optional missing review text separately from available ratings. Repeated review IDs can belong to different orders; respect the composite grain.

- Use Unknown and translation flags explicitly when grouping products. No unsupported English translations, dimensions or weights were invented.

- Use only the unique ZIP lookup for geographic joins. Retain unmatched entities and report the geographic coverage limitation. City variants and cross-state ZIP ambiguity remain unresolved; do not treat representative coordinates as precise customer locations.

- Validate numeric ranges and zero-value business meanings before selecting analytic exclusions. Dates outside the observed purchase-year range remain contextual flags.

- Keep item, payment and review grains separate before combining summaries, to prevent one-to-many join multiplication. Preserve all source transaction values.

## Completion status

The notebook executed through all cleaning, validation, exports, read-back checks, focused tests and source-integrity checks. Ten processed datasets and four reports were generated. All nine raw hashes match the initialization manifest; source DataFrames retain their original fingerprints. No SQL Server, Power BI, advanced EDA, visualizations, machine learning or Phase 3 work was implemented. Stop after Phase 2 and wait for approval.


Phase 2 complete. Stop here; Phase 3 requires approval.
